### Flux RSS

In [3]:
import os
import json
import re
import feedparser
import pandas as pd
from datetime import datetime

In [7]:
# ============================================================
# CELLULE 2 — CHARGEMENT DES BULLETINS DEPUIS LES FICHIERS LOCAUX
# Le RSS étant inaccessible, on construit la liste des bulletins
# directement depuis les dossiers data/avis/ et data/alertes/
# ============================================================
DATA_DIR = "data"

def charger_tous_les_bulletins():
    bulletins = []

    for type_bulletin, dossier in [("avis", "avis"), ("alerte", "alertes")]:
        chemin_dossier = os.path.join(DATA_DIR, dossier)
        fichiers = os.listdir(chemin_dossier)
        print(f"  {type_bulletin} : {len(fichiers)} fichiers trouvés")

        for nom_fichier in fichiers:
            chemin = os.path.join(chemin_dossier, nom_fichier)
            try:
                with open(chemin, "r", encoding="utf-8") as f:
                    data = json.load(f)

                # Date : on prend la première révision (publication initiale)
                revisions = data.get("revisions", [])
                date = revisions[0]["revision_date"] if revisions else None

                bulletins.append({
                    "id_anssi" : data.get("reference", nom_fichier),
                    "titre"    : data.get("title", "Sans titre"),
                    "date"     : date,
                    "type"     : type_bulletin,
                    "lien"     : f"https://www.cert.ssi.gouv.fr/{dossier}/{data.get('reference', '')}/"
                })

            except (json.JSONDecodeError, KeyError) as e:
                print(f"[WARN] {nom_fichier} : {e}")

    print(f"\nTotal bulletins chargés : {len(bulletins)}")
    return bulletins

bulletins_rss = charger_tous_les_bulletins()

# Aperçu
for b in bulletins_rss[:3]:
    print(b)

  avis : 4025 fichiers trouvés
  alerte : 78 fichiers trouvés

Total bulletins chargés : 4103
{'id_anssi': 'CERTFR-2025-AVI-0925', 'titre': 'Vulnérabilité dans les produits Belden', 'date': '2025-10-27T00:00:00.000000', 'type': 'avis', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2025-AVI-0925/'}
{'id_anssi': 'CERTFR-2023-AVI-0963', 'titre': 'Vulnérabilité dans les produits Cisco', 'date': '2023-11-20T00:00:00.000000', 'type': 'avis', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2023-AVI-0963/'}
{'id_anssi': 'CERTFR-2025-AVI-0119', 'titre': 'Multiples vulnérabilités dans les produits Intel', 'date': '2025-02-12T00:00:00.000000', 'type': 'avis', 'lien': 'https://www.cert.ssi.gouv.fr/avis/CERTFR-2025-AVI-0119/'}


In [8]:
# ============================================================
# CELLULE 3 — EXTRACTION DES CVE DEPUIS UN BULLETIN
# Recharge le JSON local et extrait la liste des CVE
# ============================================================

def extraire_cves(id_anssi, type_bulletin):
    """
    Recharge le fichier local du bulletin et extrait les CVE.
    Retourne une liste de strings ["CVE-xxxx", ...] ou [] si aucun CVE.
    """
    dossier = "avis" if type_bulletin == "avis" else "alertes"
    chemin  = os.path.join(DATA_DIR, dossier, id_anssi)

    try:
        with open(chemin, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"[WARN] {id_anssi} : {e}")
        return []

    # Lecture de la clé cves : liste de dicts {"name": "CVE-xxxx", "url": "..."}
    cves_brut = data.get("cves", [])
    cves = [c["name"] for c in cves_brut if "name" in c]

    # Sécurité : on filtre avec regex pour ne garder que les vrais CVE
    pattern = r"CVE-\d{4}-\d{4,7}"
    cves = [c for c in cves if re.match(pattern, c)]

    return cves

# --- Test sur les 3 premiers bulletins ---
for b in bulletins_rss[:3]:
    cves = extraire_cves(b["id_anssi"], b["type"])
    print(f"{b['id_anssi']} → {len(cves)} CVE : {cves[:3]}")

CERTFR-2025-AVI-0925 → 1 CVE : ['CVE-2024-3596']
CERTFR-2023-AVI-0963 → 1 CVE : ['CVE-2023-44487']
CERTFR-2025-AVI-0119 → 67 CVE : ['CVE-2024-38310', 'CVE-2024-25571', 'CVE-2023-34440']


In [12]:
def charger_mitre(cve_id):
    chemin = os.path.join(DATA_DIR, "mitre", cve_id)

    resultat = {
        "description" : "Non disponible",
        "cvss_score"  : None,
        "severity"    : "Non disponible",
        "cwe"         : "Non disponible",
        "cwe_desc"    : "Non disponible",
        "vendor"      : "Non disponible",
        "produit"     : "Non disponible",
        "versions"    : "Non disponible"
    }

    try:
        with open(chemin, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"[WARN] MITRE non trouvé : {cve_id}")
        return resultat
    except json.JSONDecodeError:
        print(f"[WARN] MITRE JSON invalide : {cve_id}")
        return resultat

    cna = data.get("containers", {}).get("cna", {})
    adp = data.get("containers", {}).get("adp", [])

    # --- Description ---
    descriptions = cna.get("descriptions", [])
    if descriptions:
        resultat["description"] = descriptions[0].get("value", "Non disponible")

    # --- Score CVSS ---
    # On cherche d'abord dans cna, puis dans chaque bloc adp en fallback
    def extraire_cvss(metrics):
        """Cherche cvssV3_1, cvssV3_0 ou cvssV2_0 dans une liste de metrics."""
        for m in metrics:
            for version_cle in ["cvssV3_1", "cvssV3_0", "cvssV2_0"]:
                if version_cle in m:
                    cvss = m[version_cle]
                    return cvss.get("baseScore"), cvss.get("baseSeverity", "Non disponible")
        return None, "Non disponible"

    # Tentative dans cna
    score, severity = extraire_cvss(cna.get("metrics", []))

    # Fallback dans adp si cna n'a rien
    if score is None:
        for bloc in adp:
            score, severity = extraire_cvss(bloc.get("metrics", []))
            if score is not None:
                break

    resultat["cvss_score"] = score
    resultat["severity"]   = severity

    # --- CWE ---
    problem_types = cna.get("problemTypes", [])
    if problem_types:
        descs = problem_types[0].get("descriptions", [])
        if descs:
            resultat["cwe"]      = descs[0].get("cweId", "Non disponible")
            resultat["cwe_desc"] = descs[0].get("description", "Non disponible")

    # --- Produits affectés (on prend le premier) ---
    affected = cna.get("affected", [])
    if affected:
        premier = affected[0]
        resultat["vendor"]  = premier.get("vendor", "Non disponible")
        resultat["produit"] = premier.get("product", "Non disponible")

        versions_affectees = [
            v["version"] for v in premier.get("versions", [])
            if v.get("status") == "affected"
        ]
        resultat["versions"] = ", ".join(versions_affectees) if versions_affectees else "Non disponible"

    return resultat

# --- Test sur les 3 premiers bulletins ---
for b in bulletins_rss[:3]:
    cves = extraire_cves(b["id_anssi"], b["type"])
    if cves:
        cve_id = cves[0]
        mitre  = charger_mitre(cve_id)
        print(f"\n{cve_id}")
        print(f"  CVSS     : {mitre['cvss_score']} ({mitre['severity']})")
        print(f"  CWE      : {mitre['cwe']} — {mitre['cwe_desc']}")
        print(f"  Vendor   : {mitre['vendor']} / {mitre['produit']}")
        print(f"  Versions : {mitre['versions'][:80]}")


CVE-2024-3596
  CVSS     : None (Non disponible)
  CWE      : Non disponible — CWE-328: Use of Weak Hash
  Vendor   : IETF / RFC
  Versions : 2865

CVE-2023-44487
  CVSS     : 7.5 (HIGH)
  CWE      : Non disponible — n/a
  Vendor   : n/a / n/a
  Versions : n/a

CVE-2024-38310
  CVSS     : 8.2 (HIGH)
  CWE      : Non disponible — Escalation of Privilege
  Vendor   : n/a / Intel(R) Graphics Driver software installers
  Versions : See references


In [13]:
# Diagnostic général — à supprimer après
manquants_mitre = 0
manquants_first = 0
total_cve       = 0
total_bulletins = len(bulletins_rss)

for b in bulletins_rss:
    cves = extraire_cves(b["id_anssi"], b["type"])
    for cve in cves:
        total_cve += 1
        if not os.path.exists(os.path.join(DATA_DIR, "mitre", cve)):
            manquants_mitre += 1
        if not os.path.exists(os.path.join(DATA_DIR, "first", cve)):
            manquants_first += 1

print(f"Total bulletins        : {total_bulletins}")
print(f"Total CVE uniques      : {total_cve}")
print(f"Manquants dans mitre/  : {manquants_mitre}")
print(f"Manquants dans first/  : {manquants_first}")
print(f"Couverture mitre       : {round((total_cve - manquants_mitre) / total_cve * 100, 1)}%")
print(f"Couverture first       : {round((total_cve - manquants_first) / total_cve * 100, 1)}%")

Total bulletins        : 4103
Total CVE uniques      : 125910
Manquants dans mitre/  : 0
Manquants dans first/  : 0
Couverture mitre       : 100.0%
Couverture first       : 100.0%


In [14]:
# ============================================================
# CELLULE 5 — ENRICHISSEMENT EPSS (score FIRST)
# Lit le fichier local data/first/CVE-xxxx
# Retourne la probabilité d'exploitation entre 0 et 1
# ============================================================

def charger_epss(cve_id):
    """
    Charge le score EPSS local pour un CVE donné.
    Retourne un float entre 0 et 1, ou None si indisponible.
    """
    chemin = os.path.join(DATA_DIR, "first", cve_id)

    try:
        with open(chemin, "r", encoding="utf-8") as f:
            data = json.load(f)

        epss_data = data.get("data", [])
        if epss_data:
            return float(epss_data[0]["epss"])
        return None

    except FileNotFoundError:
        print(f"[WARN] EPSS non trouvé : {cve_id}")
        return None
    except (json.JSONDecodeError, KeyError, ValueError) as e:
        print(f"[WARN] EPSS données invalides : {cve_id} — {e}")
        return None

# --- Test sur les 3 premiers bulletins ---
for b in bulletins_rss[:3]:
    cves = extraire_cves(b["id_anssi"], b["type"])
    if cves:
        cve_id = cves[0]
        epss   = charger_epss(cve_id)
        print(f"{cve_id} → EPSS : {epss}")

CVE-2024-3596 → EPSS : 0.22162
CVE-2023-44487 → EPSS : 0.94394
CVE-2024-38310 → EPSS : 0.00021


In [15]:
# ============================================================
# CELLULE 6 — BOUCLE PRINCIPALE
# Assemble tout : bulletin → CVE → MITRE + EPSS → ligne DataFrame
# Attention : 125 910 CVE à traiter, peut prendre quelques minutes
# ============================================================

lignes = []
total  = len(bulletins_rss)

for i, bulletin in enumerate(bulletins_rss):

    # Progression toutes les 500 itérations
    if i % 500 == 0:
        print(f"Progression : {i}/{total} bulletins traités...")

    # --- Extraction des CVE du bulletin ---
    cves = extraire_cves(bulletin["id_anssi"], bulletin["type"])

    if not cves:
        # Bulletin sans CVE → une ligne avec les infos ANSSI uniquement
        lignes.append({
            "id_anssi"   : bulletin["id_anssi"],
            "titre"      : bulletin["titre"],
            "type"       : bulletin["type"],
            "date"       : bulletin["date"],
            "lien"       : bulletin["lien"],
            "cve"        : None,
            "cvss_score" : None,
            "severity"   : None,
            "cwe"        : None,
            "cwe_desc"   : None,
            "epss"       : None,
            "description": None,
            "vendor"     : None,
            "produit"    : None,
            "versions"   : None
        })
        continue

    # --- Pour chaque CVE : enrichissement MITRE + EPSS ---
    for cve_id in cves:

        mitre = charger_mitre(cve_id)
        epss  = charger_epss(cve_id)

        lignes.append({
            "id_anssi"   : bulletin["id_anssi"],
            "titre"      : bulletin["titre"],
            "type"       : bulletin["type"],
            "date"       : bulletin["date"],
            "lien"       : bulletin["lien"],
            "cve"        : cve_id,
            "cvss_score" : mitre["cvss_score"],
            "severity"   : mitre["severity"],
            "cwe"        : mitre["cwe"],
            "cwe_desc"   : mitre["cwe_desc"],
            "epss"       : epss,
            "description": mitre["description"],
            "vendor"     : mitre["vendor"],
            "produit"    : mitre["produit"],
            "versions"   : mitre["versions"]
        })

print(f"\nTerminé — {len(lignes)} lignes générées")

Progression : 0/4103 bulletins traités...
Progression : 500/4103 bulletins traités...
Progression : 1000/4103 bulletins traités...
Progression : 1500/4103 bulletins traités...
Progression : 2000/4103 bulletins traités...
Progression : 2500/4103 bulletins traités...
Progression : 3000/4103 bulletins traités...
Progression : 3500/4103 bulletins traités...
Progression : 4000/4103 bulletins traités...

Terminé — 126101 lignes générées


In [16]:
# ============================================================
# CELLULE 7 — CONSTRUCTION DU DATAFRAME + EXPORT CSV
# ============================================================

df = pd.DataFrame(lignes)

# --- Nettoyage des types ---

# Date en datetime
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")

# CVSS et EPSS en numérique
df["cvss_score"] = pd.to_numeric(df["cvss_score"], errors="coerce")
df["epss"]       = pd.to_numeric(df["epss"],       errors="coerce")

# --- Aperçu ---
print(f"Shape : {df.shape}")
print(f"\nTypes des colonnes :")
print(df.dtypes)
print(f"\nAperçu des 3 premières lignes :")
df.head(3)

Shape : (126101, 15)

Types des colonnes :
id_anssi                       str
titre                          str
type                           str
date           datetime64[us, UTC]
lien                           str
cve                            str
cvss_score                 float64
severity                       str
cwe                            str
cwe_desc                       str
epss                       float64
description                    str
vendor                         str
produit                        str
versions                       str
dtype: object

Aperçu des 3 premières lignes :


,id_anssi,titre,type,date,lien,cve,cvss_score,severity,cwe,cwe_desc,epss,description,vendor,produit,versions
0,CERTFR-2025-AVI-0925,Vulnérabilité dans les produits Belden,avis,2025-10-27 00:00:00+00:00,https://www.cert.ssi.gouv.fr/avis/CERTFR-2025-...,CVE-2024-3596,NaN,Non disponible,Non disponible,CWE-328: Use of Weak Hash,0.22162,RADIUS Protocol under RFC 2865 is susceptible ...,IETF,RFC,2865
1,CERTFR-2023-AVI-0963,Vulnérabilité dans les produits Cisco,avis,2023-11-20 00:00:00+00:00,https://www.cert.ssi.gouv.fr/avis/CERTFR-2023-...,CVE-2023-44487,7.5,HIGH,Non disponible,n/a,0.94394,The HTTP/2 protocol allows a denial of service...,n/a,n/a,n/a
2,CERTFR-2025-AVI-0119,Multiples vulnérabilités dans les produits Intel,avis,2025-02-12 00:00:00+00:00,https://www.cert.ssi.gouv.fr/avis/CERTFR-2025-...,CVE-2024-38310,8.2,HIGH,Non disponible,Escalation of Privilege,0.00021,Improper access control in some Intel(R) Graph...,n/a,Intel(R) Graphics Driver software installers,See references


In [17]:
# ============================================================
# CELLULE 8 — EXPORT CSV
# ============================================================

chemin_csv = "data/bulletins_enrichis.csv"
df.to_csv(chemin_csv, index=False, encoding="utf-8-sig")
print(f"CSV exporté → {chemin_csv}")
print(f"Taille : {round(os.path.getsize(chemin_csv) / 1024 / 1024, 1)} Mo")

CSV exporté → data/bulletins_enrichis.csv
Taille : 161.0 Mo


### Identification des CVE 